In [ ]:
from pathlib import Path
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path('..').resolve()
AIRR = ROOT / 'airr_yfv19'
CACHE = ROOT / 'deep_data_audit' / 'cache'
OUT = ROOT / 'coursework_submission_notebooks' / 'outputs' / '01_qc'
OUT.mkdir(parents=True, exist_ok=True)
DONORS = ['P1', 'P2', 'Q1', 'Q2', 'S1', 'S2']
NAME_RE = re.compile('^(P1|P2|Q1|Q2|S1|S2)_(pre0|\\d+)_(F\\d+)\\.airr\\.tsv\\.gz$')


In [ ]:
rows = []
for path in sorted(AIRR.glob('*.airr.tsv.gz')):
    m = NAME_RE.match(path.name)
    if not m:
        continue
    (donor, day, rep) = (m.group(1), m.group(2), m.group(3))
    n_rows = 0
    total = 0
    for chunk in pd.read_csv(path, sep='\t', compression='gzip', usecols=['duplicate_count'], chunksize=200000):
        n_rows += len(chunk)
        total += int(chunk['duplicate_count'].sum())
    rows.append({'file': path.name, 'donor': donor, 'day': day, 'replicate': rep, 'n_rows': n_rows, 'total_reads': total})
inventory = pd.DataFrame(rows).sort_values(['donor', 'day', 'replicate'])
inventory.to_csv(OUT / 'inventory_airr_files.csv', index=False)
inventory.head(20)


In [ ]:
qc_rows = []
for donor in DONORS:
    for tp in ['0', '15']:
        path = CACHE / f'{donor}_{tp}_F1.parquet'
        df = pd.read_parquet(path)
        qc_rows.append({'donor': donor, 'timepoint': int(tp), 'n_unique_cdr3nt': df['cdr3_nt'].nunique(), 'total_reads': int(df['duplicate_count'].sum()), 'n_unique_cdr3aa': df['cdr3_aa'].nunique()})
qc = pd.DataFrame(qc_rows)
qc.to_csv(OUT / 'depth_day0_day15.csv', index=False)
qc


In [ ]:
(fig, axes) = plt.subplots(1, 2, figsize=(10, 4))
wide_reads = qc.pivot(index='donor', columns='timepoint', values='total_reads')
wide_uniq = qc.pivot(index='donor', columns='timepoint', values='n_unique_cdr3nt')
x = np.arange(len(DONORS))
w = 0.35
axes[0].bar(x - w / 2, wide_reads[0].loc[DONORS] / 1000000.0, width=w, label='Day 0')
axes[0].bar(x + w / 2, wide_reads[15].loc[DONORS] / 1000000.0, width=w, label='Day 15')
axes[0].set_xticks(x)
axes[0].set_xticklabels(DONORS)

axes[0].legend()
axes[1].bar(x - w / 2, wide_uniq[0].loc[DONORS] / 100000.0, width=w, label='Day 0')
axes[1].bar(x + w / 2, wide_uniq[15].loc[DONORS] / 100000.0, width=w, label='Day 15')
axes[1].set_xticks(x)
axes[1].set_xticklabels(DONORS)

axes[1].legend()
plt.tight_layout()
plt.savefig(OUT / 'figure_depth_day0_day15.png', dpi=150, bbox_inches='tight')
plt.show()
